# Predicting Heart Disease
Playground Series - Season 6 Episode 2

Predicting heart disease likelihood enables early interventions that save lives and ease healthcare burdens. By analyzing key clinical indicators like age, chest pain type, blood pressure, cholesterol, and stress test results, etc., models can assist identifying high-risk patients for targeted prevention. These insights guide personalized care plans and optimize population-level cardiovascular outcomes.


Yao Yan, Walter Reade, Elizabeth Park. Predicting Heart Disease. https://kaggle.com/competitions/playground-series-s6e2, 2026. Kaggle.

In [ ]:
import pandas as pd
import numpy as np
import optuna
import cupy as cp
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier
from sklearn.ensemble import VotingClassifier

try:
    from IPython.core.magic import register_cell_magic
    @register_cell_magic
    def skip(line, cell): return
except:
    pass

## About the data

| Column | Description |
|--------|-------------|
| **Age** 🧓 | Age of the patient (in years) |
| **Sex** 🚹 | Gender of the patient (1 = Male, 0 = Female) |
| **Chest pain type** 💔 | Type of chest pain: 1=Typical angina, 2=Atypical angina, 3=Non-anginal pain, 4=Asymptomatic |
| **BP** 💉 | Resting blood pressure (mm Hg) |
| **Cholesterol** 🧈 | Serum cholesterol level (mg/dL) |
| **FBS over 120** 🍬 | Fasting blood sugar > 120 mg/dL (1 = True, 0 = False) |
| **EKG results** 📈 | Resting electrocardiogram results: 0=Normal, 1=ST-T wave abnormality, 2=Left ventricular hypertrophy |
| **Max HR** ❤️ | Maximum heart rate achieved |
| **Exercise angina** 🏃 | Exercise-induced angina (1 = Yes, 0 = No) |
| **ST depression** 📉 | ST depression induced by exercise relative to rest |
| **Slope of ST** ⛰️ | Slope of the peak exercise ST segment |
| **Number of vessels fluro** 🩸 | Number of major vessels (0–3) colored by fluoroscopy |
| **Thallium** 🧬 | Thallium stress test result (categorical medical indicator) |
| **Heart Disease** 🎯 | Target: Presence (❤️) or Absence (💚) of heart disease |

In [ ]:
train = pd.read_csv('/kaggle/input/playground-series-s6e2/train.csv', index_col="id")
test = pd.read_csv('/kaggle/input/playground-series-s6e2/test.csv', index_col="id")
submission_sample = pd.read_csv('/kaggle/input/playground-series-s6e2/sample_submission.csv')

In [ ]:
def downcasting(data: pd.DataFrame, verbose: bool = True) -> pd.DataFrame:
    mem_before = data.memory_usage().sum() / 1024**2
    if verbose:
        print(f"Memory usage of dataframe is {mem_before:.2f} MB")
    
    for col in data.select_dtypes(include=["number"]).columns:
        if pd.api.types.is_integer_dtype(data[col]):
            data[col] = pd.to_numeric(data[col], downcast="integer")
        elif pd.api.types.is_float_dtype(data[col]):
            data[col] = pd.to_numeric(data[col], downcast="float")
    
    mem_after = data.memory_usage().sum() / 1024**2
    if verbose:
        print(f"Memory usage after optimization is: {mem_after:.2f} MB")
        print(f"Decreased by {(100 * (mem_before - mem_after) / mem_before):.1f}%\n")
    
    return data

print("Train train:")
train = downcasting(train)
print("Test train:")
test = downcasting(test)

## Feature Engineering

I added 6 domain-inspired interaction terms to capture clinical synergies that tree splits might miss on this small UCI dataset.

In [ ]:
def create_heart_interactions(df):
    df_out = df.copy()
    
    # 1. Age × Sex (gender-specific risk)
    df_out['Age_Sex'] = df_out['Age'] * df_out['Sex']
    
    # 2. Chest pain × ST depression (angina + ischemia)
    df_out['ChestPain_STdep'] = df_out['Chest pain type'] * df_out['ST depression']
    
    # 3. Cholesterol × Max HR (lipids + exercise capacity)
    df_out['Chol_MaxHR'] = df_out['Cholesterol'] * df_out['Max HR']
    
    # 4. EKG × Slope of ST (ECG + ST dynamics)
    df_out['EKG_STslope'] = df_out['EKG results'] * df_out['Slope of ST']
    
    # 5. BP / Max HR (hemodynamic stress ratio)
    df_out['BP_over_MaxHR'] = df_out['BP'] / (df_out['Max HR'] + 1)  # +1 avoids div/0
    
    # 6. Age × Vessels (atherosclerosis progression)
    df_out['Age_Vessels'] = df_out['Age'] * df_out['Number of vessels fluro']
    
    return df_out

train = create_heart_interactions(train)
test = create_heart_interactions(test)

In [ ]:
target = "Heart Disease"

train[target] = train[target].map({
    'Presence': 1,
    'Absence': 0 
})

target = 'Heart Disease'
X = train.drop(target, axis=1)
y = train[target]

# Modelling

For this classification task, I'm using XGBoost because tree-based gradient boosting models excel at capturing nonlinear relationships and complex feature interactions in tabular data Its built-in regularization and early stopping also make it robust against overfitting. 

To push accuracy further, I'm ensembling it with LightGBM and CatBoost. Their complementary splitting strategies and inductive biases reduce variance and blind spots, giving us a more stable, high-performing prediction.

In [ ]:
xgb = XGBClassifier(random_state=42, eval_metric='auc')
cv_scores = cross_val_score(xgb, X, y, cv=5, scoring='roc_auc')
print(f"✅ CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42
)

xgb = XGBClassifier(
    random_state=42, 
    eval_metric='auc',
    early_stopping_rounds=50,  
    n_estimators=1000         
)
xgb.fit(
    X_train, y_train,
    eval_set=[(X_valid, y_valid)],
    verbose=False
)

valid_auc = roc_auc_score(y_valid, xgb.predict_proba(X_valid)[:, 1])
print(f"📊 Validation AUC: {valid_auc:.4f}")

In [ ]:
%%skip
def objective(trial):
    params = {
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.2),
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'subsample': trial.suggest_float('subsample', 0.7, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.7, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0, 10),
        'reg_lambda': trial.suggest_float('reg_lambda', 0, 10),
    }
    
    model = XGBClassifier(**params, random_state=42, eval_metric='auc')
    scores = cross_val_score(model, X, y, cv=5, scoring='roc_auc')
    
    return scores.mean()

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50) 

print(f"Best CV AUC: {study.best_value:.4f}")
print("Best params:", study.best_params)

final_xgb = XGBClassifier(**study.best_params, random_state=42, eval_metric='auc')
cv_scores = cross_val_score(final_xgb, X, y, cv=5, scoring='roc_auc')
print(f"✅ Final CV AUC: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")

In [ ]:
# no interactions
best_params_1 = {
    'max_depth': 3,
    'learning_rate': 0.0985,
    'n_estimators': 957,
    'subsample': 0.8653,
    'colsample_bytree': 0.8107,
    'reg_alpha': 7.006,
    'reg_lambda': 3.049
}

# with interaction terms
best_params_2 = {
    'max_depth': 3,
    'learning_rate': 0.133,
    'n_estimators': 566,
    'subsample': 0.986,
    'colsample_bytree': 0.985,
    'reg_alpha': 2.343,
    'reg_lambda': 6.435
}

In [ ]:
ensemble = VotingClassifier([
    ('xgb', XGBClassifier(**best_params_2)),
    ('lgb', LGBMClassifier(verbose=-1, n_estimators=500)),  
    ('cat', CatBoostClassifier(verbose=False, iterations=500))  
], voting='soft')

## Prediction

In [ ]:
ensemble.fit(X, y)

test_preds = ensemble.predict_proba(test)[:, 1]

submission = pd.DataFrame({
    'id': test.index, 
    'Heart Disease': test_preds
})
submission.to_csv('submission.csv', index=False)